<a href="https://colab.research.google.com/github/ananaysaharan/DL_Basics/blob/main/embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install tiktoken torch

In [2]:
import torch
import tiktoken

# Define vocabulary size and embedding dimensions
vocab_size = 50000
embedding_dim = 768

# Create a random embedding matrix
# Using torch.nn.Embedding for a typical embedding layer structure
embedding_matrix = torch.nn.Embedding(vocab_size, embedding_dim)

print(f"Embedding matrix created with shape: {embedding_matrix.weight.shape}")

Embedding matrix created with shape: torch.Size([50000, 768])


In [3]:
# Initialize tiktoken tokenizer
# Using 'cl100k_base' which is common for OpenAI models like GPT-3.5/4
enc = tiktoken.get_encoding("cl100k_base")

# Example string
example_string = "hello world"

# Tokenize the string
tokens = enc.encode(example_string)

print(f"Original string: '{example_string}'")
print(f"Encoded tokens (integers): {tokens}")
print(f"Decoded tokens: {[enc.decode_single_token_bytes(token) for token in tokens]}")

Original string: 'hello world'
Encoded tokens (integers): [15339, 1917]
Decoded tokens: [b'hello', b' world']


In [4]:
# Perform embedding lookup
# Convert tokens list to a torch.LongTensor for lookup
tokens_tensor = torch.LongTensor(tokens)

# Get embeddings for the tokens
looked_up_embeddings = embedding_matrix(tokens_tensor)

print(f"\nLooked-up embeddings shape: {looked_up_embeddings.shape}")
print("Embeddings for 'hello world' tokens:")
for i, token_id in enumerate(tokens):
    print(f"  Token ID {token_id} (value: {enc.decode_single_token_bytes(token_id)}):\n{looked_up_embeddings[i].detach().numpy()[:5]}...") # Displaying first 5 dimensions


Looked-up embeddings shape: torch.Size([2, 768])
Embeddings for 'hello world' tokens:
  Token ID 15339 (value: b'hello'):
[-2.3769429   1.4797884  -0.17901053 -1.0039506   0.24905583]...
  Token ID 1917 (value: b' world'):
[ 0.88827986 -1.584247    0.16090019 -0.22631912  2.6828916 ]...


### Using Gensim Word2Vec for Pre-trained Embeddings

Now, let's explore embeddings using a pre-trained Word2Vec model from the `gensim` library. We'll download a common model and then use it to find similar words and perform vector arithmetic.

**Note:** The `word2vec-google-news-300` model is large (~3.6 GB compressed) and may take some time to download.

In [10]:
# Install gensim
!pip install gensim

In [11]:
import gensim.downloader as api

# Load the pre-trained Word2Vec model
# This model is ~3.6GB compressed and will take time to download
print("Downloading word2vec-google-news-300 model... This might take a few minutes.")
word_vectors = api.load('word2vec-google-news-300')
print("Model loaded successfully!")

[==================================================] 100.0% 1662.8/1662.8MB downloaded
Model loaded successfully!


In [12]:
print("\nWords similar to 'king':")
# Find the 10 most similar words to 'king'
similar_to_king = word_vectors.most_similar('king', topn=10)
for word, similarity in similar_to_king:
    print(f"  {word}: {similarity:.4f}")


Words similar to 'king':
  kings: 0.7138
  queen: 0.6511
  monarch: 0.6413
  crown_prince: 0.6204
  prince: 0.6160
  sultan: 0.5865
  ruler: 0.5798
  princes: 0.5647
  Prince_Paras: 0.5433
  throne: 0.5422


In [13]:
print("\nVector arithmetic: 'king - man + woman'")
# Perform vector arithmetic: king - man + woman
# This is often used to find analogies, e.g., 'King is to Man as Queen is to Woman'
result_vector = word_vectors.most_similar(positive=['king', 'woman'], negative=['man'], topn=1)

for word, similarity in result_vector:
    print(f"  The word most similar to 'king - man + woman' is '{word}' with similarity {similarity:.4f}")


Vector arithmetic: 'king - man + woman'
  The word most similar to 'king - man + woman' is 'queen' with similarity 0.7118
